In [1]:
# İngilizce NER

import spacy
nlp_en = spacy.load("en_core_web_sm")

text = """
    Apple Inc. was founded by Steve Jobs in California on April 1, 1976,
The company reported $383 revenue in 2023.
Tim Cook now leads the company from Cupertino."""

doc = nlp_en(text)

# Kullacağımız alanlar/fonksiyonlar:
# doc.ents      -> Belgede bulunan varlıkların listesi (her biri bir 'ent')
# ent.text      -> Varlığın metni       (ör. 'Apple Inc')
# ent.label_    -> varlığın TİPİ, string  (ör. 'ORG') <- sondaki _ önemli:
#                    label_ string verir; label (alt cizgisiz) sayı kodu verir
# spacy.explain(kod)    -> tip kodunun açıklaması (ör. 'GPE' -> 'Countries, cities, states')

print(f"{'Text':<25} {'Label':<10} {'Explain'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<10} {spacy.explain(ent.label_)}")

Text                      Label      Explain
------------------------------------------------------------
Apple Inc.                ORG        Companies, agencies, institutions, etc.
Steve Jobs                PERSON     People, including fictional
California                GPE        Countries, cities, states
April 1, 1976             DATE       Absolute or relative dates or periods
383                       MONEY      Monetary values, including unit
2023                      DATE       Absolute or relative dates or periods
Tim Cook                  PERSON     People, including fictional
Cupertino                 GPE        Countries, cities, states


In [2]:
# Görselleştirme - dsplaCy ile varlılkları renkli HTML olarak çiz
from spacy import displacy # ilk kullanımda import 

# jupyter = False -> HTML string döner (None değil); nbconvert altında güvenli

html = displacy.render(doc, style= "ent", jupyter= False)
with open('./new_output.html', 'w') as f:
    f.write(html)
print("Görsel: ./ner_output.html")

Görsel: ./ner_output.html


# Türkçe NER -- Dürüst Durum

- spaCy'nin resmi bir Türkçe pipeline'ı yoktur. Bu yüzden spacy.load('tr_core_news_sm') çalışmaz ve python -m spacy download tr_cıore_news_sm komuta da bir işe yaramaz.

1. Stanza - resmi Türkçe desteği var:
    - import stanza
    stanza.download('tr')
    nlp_tr = stanza.Pipeline("tr", precessors = "tokenize,ner")
    doc = nlp_tr("Ahmet Yılmaz Ankara'da Turkcell yöneticileriyle görüştü.")
    for sent in doc.sentences:
        for ent in sent.ents:
            print(ent.text, ent.type)

2. HuggingFace transformers (BERTurk tabanlı) - savasy/bert-base-turkish-ner-cased
    - pip install transformers torch
    from transformers import pipeline
    ner = pipeline("ner", model = "savasy/bert-turkish-ner-cased", aggregation_strategy = "simple")
    print(new("Ahmet Yılmaz Ankara'da Turkcell yöneticileriyle görüştü."))

In [3]:
try:
    nlp_tr = spacy.load("tr_core_news_sm") # spacy'de resmi tÜRKÇE MODEL YOK

except OSError:
    print("spacy'nin resmi Türkçe pipeline'ı yok.")
    print("Türkçe NER için STANZA ('TR') veya HuggingFace 'savasy/bert-base-turkish-ner-cased' kullanın")

spacy'nin resmi Türkçe pipeline'ı yok.
Türkçe NER için STANZA ('TR') veya HuggingFace 'savasy/bert-base-turkish-ner-cased' kullanın


In [4]:
news = """
Tesla increased productions at its Shanghai factory by 20 percent in the fourth quarter of 2024,
Chief executive Elon musk said the company expects strong demand in 2025.
Apple earned $15 billion in revenue from in revenue from İphone sales last year, according to Tim Cook.
Microsoft and OpenAI announces new AI tools on the Azure platform their offices in Seattle.
"""

doc = nlp_en(news)
# Varlık tiplerini say (doc.ents üzerinde gezinmek = veri işleme, demo döngüsü değil)
from collections import Counter
entity_types = Counter([ent.label_ for ent in doc.ents])
print("Varlık tipleri", dict(entity_types))

# Şirketler (ORG)
# Kişiler
orgs = []
people = []
dates = []
for ent in doc.ents:
    if ent.label_ == 'ORG':
        orgs.append(ent.text)
    elif ent.label_ == 'PERSON':
        people.append(ent.text)
    elif ent.label_ == "DATE":
        dates.append(ent.text)
print(f"Şirketler: {orgs}")
print(f"Kişiler: {people}")
print(f"Tarihler: {dates}")



Varlık tipleri {'GPE': 3, 'PERCENT': 1, 'DATE': 3, 'ORG': 4, 'MONEY': 1, 'PERSON': 1}
Şirketler: ['Apple', 'İphone', 'Microsoft', 'OpenAI']
Kişiler: ['Tim Cook']
Tarihler: ['the fourth quarter of 2024', '2025', 'last year']


In [8]:
# CUSTOM NER (ileri konu)
# Spesifik bir domain için (ör. ilaç isimleri, kod sembolleri)
# 1. Training data hazırla: [(text, {"entities": [start,end,LABEL]}), ...]
# 2. spacy train config -> python -m spacy train config.cfg
#
# Hızlı prototip: Matcher kullan (kural tabanlı)

from spacy.matcher import Matcher # ilk kullanımda import

matcher = Matcher(nlp_en.vocab)
pattern = [{"LOWER": "iphone"}, {"IS_DIGIT": True}] # 'iphone' + sayı
matcher.add("IPHONE_MODEL", [pattern])

doc = nlp_en("I bought an iPhone 15 and iPhone 13 yesterday")
matches = matcher(doc)
print(matches)

# Demo: iki eşleşmeyi açıkça yazdır (döngüyü açtık)
match_id_0, start_0, end_0 = matches[0]
print(f"Match: {doc[start_0:end_0]}")
match_id_1,start_1, end_1 = matches[1]
print(f"Match: {doc[start_1:end_1]}")


[(8124601010157298845, 3, 5), (8124601010157298845, 6, 8)]
Match: iPhone 15
Match: iPhone 13


# İleri (opsiyonel): Matcher eşleşmesini gerçek varlığa çevirme

- Matcher bir kalıbı bulur ama doc.ents'ee eklemez. Modelde olmayan kendi varlıklarını (ürün adı, ilaç ismi, ürün kodu...) çıkarmak istersen: eşleşmeyi Span yap -> doc.set_ents ile doc'a yaz (filter_spans çakışanları eler). Aşağıda önce/sonra farkı:

In [10]:
# MATCHER -> doc.ents: bulunan eşleşmeyi GERÇEK varlığa dönüştür
# MATCHER tek başına span bulur ama doc.ents'e eklemez. Onu biz ekleriz
from spacy.tokens import Span
from spacy.util import filter_spans

doc = nlp_en("I bought an iPhone 15 and iPhone 13 yesterday from Apple")

print("ÖNCE (varsayılan):", [(e.text, e.label_) for e in doc.ents])

# Matcher eşleşmeleri PRODUCT etiketli Span'lere çevir
matches = matcher(doc)
custom_spans = [Span(doc, start, end, label= "PRODUCT") for _, start, end in matches]

# Var olan varlıklar + custom span'leri birleştir, çakışmaları temizle, doc'a yaz
doc.set_ents(filter_spans(list(doc.ents) + custom_spans))

# Sonra: artık 'İphone 15' ve 'iPhone 13' düzgünce PRODUCT olarak çıkıyor
print("SONRA (custom): ", [(e.text,e.label_) for e in doc.ents])

print("-" * 40)
for ent in doc.ents:
    print(f"{ent.text:<12} {ent.label_}")

ÖNCE (varsayılan): [('13', 'CARDINAL'), ('yesterday', 'DATE'), ('Apple', 'ORG')]
SONRA (custom):  [('iPhone 15', 'PRODUCT'), ('iPhone 13', 'PRODUCT'), ('yesterday', 'DATE'), ('Apple', 'ORG')]
----------------------------------------
iPhone 15    PRODUCT
iPhone 13    PRODUCT
yesterday    DATE
Apple        ORG
